# data incident responder: evaluation run

Everything runs in this notebook: the model is served by vLLM on the GPU, and the warehouse, dbt, the MCP server
and the agents run next to it. Runtime: A100 (80 GB) with high RAM.

Optional: add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` as Colab secrets to trace every run.

Results are written to Drive after each incident. If the runtime drops, run all again and finished incidents are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/arihaaant/data-incident-responder.git /content/responder
%cd /content/responder
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

In [ ]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

## serve the model

In [ ]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

## warehouse

In [ ]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
# stop here if anything is broken, so a bad environment never produces scorecards
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "tests failed, fix before running"

## runs: the responder and both baselines on the 48 development incidents

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "run2"  # a new name for every evaluation, since finished incidents in a folder are skipped
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

In [ ]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.run --system {system} --split dev --workers 6 --out {out}/dev-{system}

## results

In [ ]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.report {out}/dev-{system}